# Get current price data to test the model on real stocks

In [2]:
import numpy as np
import yfinance as yf
from sklearn.covariance import LedoitWolf
from scipy.stats import gmean
import pandas as pd

Selecting 8 stocks in the sp500 and calculating their covariance using the LedoitWolf method

In [40]:
start = '2023-09-24'
end = '2026-03-04'
tickers = ['^FTSE', '^GSPC', '^DJI', '^IXIC', '^N225', '^HSI', '^STOXX50E', '^FCHI']

def create_cov(tickers, start, end):
    # download the data
    prices = yf.download(tickers, start, end, auto_adjust=False)['Adj Close']

    # 1. Prepare your data (dropp_puting NAs is crucial for the solver)
    log_returns = np.log(prices / prices.shift(1)).dropna()

    # 2. Initialize the Ledoit-Wolf estimator
    lw = LedoitWolf()

    # 3. Fit the model to your returns 
    # Note: sklearn expects (observations, features), which matches your log_returns layout
    lw.fit(log_returns)

    # 4. Extract the shrunKcovariance matrix
    # This is the "cleaner" version of your cov_matrix
    shrunk_cov = lw.covariance_

    # 5. Annualize (same as your original code)
    shrunk_cov_annual = shrunk_cov * 252

    prices.to_csv("data/etf_prices_04032026.csv")
    pd.DataFrame(shrunk_cov_annual, columns=tickers, index=tickers).to_csv("data/etf_covariance_matrix_04032026.csv")


create_cov(tickers, start, end)


etf_shrunk_cov_annual = np.array(pd.read_csv("data/etf_covariance_matrix_04032026.csv", index_col=0))
etf_prices = pd.read_csv("data/etf_prices_04032026.csv", index_col='Date')

[*********************100%***********************]  8 of 8 completed


In [41]:
# get current prices
S0_etf = etf_prices.iloc[-1, :]
S0_etf = np.array(S0_etf)

# get geometric average of the stocKprices
S0_etf_geo_mean = gmean(S0_etf)

# choose strike price
K_etf = 1.01 * S0_etf_geo_mean

print(f'Strike: {K_etf}')

Strike: 16588.014103785066


Get the dividend yields for each stock

Create parameter dictionary

In [20]:
r = 0.0375             # current BOE rate
div = 0                # no dividends on ETFs
T = 0.25               # exercise time
d = len(tickers)       # number of assets

p_call = {}
p_call['strike'] = K_etf
p_call['rate'] = r     
p_call['dividend'] = div
p_call['expiration'] = T
p_call['dim'] = d
p_call['S0'] = S0_etf
p_call['volatility'] = None
p_call['correlation'] = None
p_call['covariance'] = etf_shrunk_cov_annual
p_call['numTimeStep'] = 50
p_call['callput'] = 'call'

p_put = p_call.copy()
p_put['callput'] = 'put'

In [ ]:
# Import the model
import glsm_geobasketcall

Run the G-LSM model to estimate the price of such an option

In [15]:
# Call the main function
glsm_call_etf, _ = glsm_geobasketcall.main(p_call)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 302.9683
---------------------------------------------
run trial no.2, price = 302.1243
---------------------------------------------
run trial no.3, price = 305.1121
---------------------------------------------
run trial no.4, price = 290.1128
---------------------------------------------
run trial no.5, price = 331.3216
---------------------------------------------
run trial no.6, price = 337.6972
---------------------------------------------
run trial no.7, price = 309.4148
---------------------------------------------
run trial no.8, price = 330.9980
---------------------------------------------
run trial no.9, price = 292.4037
---------------------------------------------
run trial no.10, price = 295.3473
---------------------------------------------

Mean price: 309.7500
Std dev:    16.4851


Now price the call, we checKthe number with the put call parity

In [16]:
# Call the main function
glsm_put_etf, _ = glsm_geobasketcall.main(p_put)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 388.4182
---------------------------------------------
run trial no.2, price = 374.6754
---------------------------------------------
run trial no.3, price = 364.5733
---------------------------------------------
run trial no.4, price = 353.6093
---------------------------------------------
run trial no.5, price = 383.7075
---------------------------------------------
run trial no.6, price = 379.0566
---------------------------------------------
run trial no.7, price = 397.5819
---------------------------------------------
run trial no.8, price = 367.3459
---------------------------------------------
run trial no.9, price = 370.1527
---------------------------------------------
run trial no.10, price = 373.7918
---------------------------------------------

Mean price: 375.2912
Std dev:    11.9639


### Risk-Neutral Pricing of Geometric Mean Options

We assume that, under the risk-neutral probability measure:
$$dS_i(t) = S_i(t)(rdt + \sigma_i dW_i(t)), \quad \text{for } i=1, \dots, n$$

Where:
* **$r$** is the interest rate.
* **$\sigma_i$** is the volatility.
* **$\{W_i(t), t \geq 0\}$** is a standard Brownian motion such that $d\langle W_i, W_j \rangle(t) = \rho_{i,j}dt$ for $i \neq j$.

Note that it is assumed that there is no dividend yield here, the analysis does not work if there is a divident. I will address that later.

Then, the geometric mean is given by:
$$\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} = \left(\prod_{i=1}^{n} S_i(0)\right)^{\frac{1}{n}} \exp\left( \left(r - \frac{1}{2n} \sum_{i=1}^{n} \sigma_i^2\right)T + \frac{1}{n} \sum_{i=1}^{n} \sigma_i W_i(T) \right)$$

Let the aggregate volatility $\sigma$ and the standard normal variable $\xi$ be defined as:
$$\sigma = \sqrt{\frac{1}{n^2} \left( \sum_{i=1}^{n} \sigma_i^2 + 2\sum_{i < j} \rho_{i,j} \sigma_i \sigma_j \right)}, \quad \xi = \frac{\frac{1}{n} \sum_{i=1}^{n} \sigma_i W_i(T)}{\sigma \sqrt{T}}$$

Since we have a covariance matrix ($Sigma$) instead of volatility ($\sigma_i$) and correlations ($\rho_{i,j}$), we can use the elemnts of the covariance matrix are defined as $\Sigma_{i,j} = \sum_{i,j}\sigma_i\sigma_j\phi_{i,j}$ to rewrite the equation for $\sigma$ as

$$\sigma = \sqrt{\frac{1}{n^2}\sum_{i=1}^{n}\sum_{j=1}^{n}\Sigma_{i,j}}.$$

Then, $\xi \sim N(0,1)$. Let:
$$F = \left(\prod_{i=1}^{n} S_i(0)\right)^{\frac{1}{n}} \exp\left( \left(r - \frac{1}{2n} \sum_{i=1}^{n} \sigma_i^2 + \frac{1}{2}\sigma^2\right)T \right)$$

The geometric mean at time $T$ can be expressed as:
$$\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} = F e^{-\frac{1}{2}\sigma^2 T + \sigma \sqrt{T} \xi}$$

Following the derivation of the **Black-Scholes formula**, the expected payoff is:
$$E\left[\max\left(\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} - K, 0\right)\right] = F\Phi(d_1) - K\Phi(d_2)$$

Where $\Phi$ is the standard normal CDF, and:
$$d_1 = \frac{\ln(F/K) + \frac{1}{2}\sigma^2 T}{\sigma \sqrt{T}}, \quad d_2 = d_1 - \sigma \sqrt{T}$$

The option value is then:
$$C = e^{-rT} [F\Phi(d_1) - K\Phi(d_2)].$$

Using the put-call parity, we can derive the expression for the price of the geometric put option.

Put-call partiy:
$$C - P = e^{-rT}(F-K)$$

Rearranging for P, and substituting in C from above we find

\begin{align}
P &= e^{-rT} [F\Phi(d_1) - K\Phi(d_2)] - e^{-rT}(F-K)\\
P &= e^{-rT}[K(1-\Phi(d_2)) - F(1-\Phi(d_1))].
\end{align}

We now use the fact that $1-\Phi(a) = \Phi(-a)$ to find the value of the put option is

$$P=e^{-rT}[K\Phi(-d_2) - F\Phi(-d_1)].

In [44]:
# used to find the cumulative distribution values
from scipy.stats import norm

# calculating the price of the call and the put with put call parity

def put_call_parity(Sigma, S0, K, r, d, T):

    # breaking up the calculations
    sigmas_sqr = np.diag(Sigma)
    sum_sig_sqr = np.sum(sigmas_sqr)
    sigma = np.sqrt(1/d**2 * np.sum(etf_shrunk_cov_annual))


    # term in the exponential for F
    exp_term = (r - 1/(2*d)*sum_sig_sqr + 0.5*sigma**2) * T

    # calculate F
    F = S0 * np.exp(exp_term)

    # calculate d1 and d2
    d1 = (np.log(F/K_etf) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)

    # calcualte the values from the cumulative distribution function
    phi_d1 = norm.cdf(d1); phi_md1 = norm.cdf(-d1)
    phi_d2 = norm.cdf(d2); phi_md2 = norm.cdf(-d2)

    # prices
    C = np.exp(-r*T) * (F*phi_d1 - K*phi_d2)
    P = np.exp(-r*T) * (K*phi_md2 - F*phi_md1)

    return C, P

C, P = put_call_parity(etf_shrunk_cov_annual, S0_etf_geo_mean, K_etf, r, d, T)

print(f'Call price: {C}')
print(f'Put price: {P}')


Call price: 307.2259186925064
Put price: 359.14334164802307


# Error in G-LSM method

In [43]:
error_call = abs(C - glsm_call_etf) / abs(C)
error_put = abs(P - glsm_put_etf) / abs(P)

print(f'G-LSM call price: {glsm_call_etf:.2}')
print(f'G-LSM put price: {glsm_put_etf:.2}')

print(f'Call error: {error_call*100:.2}%')
print(f'Put error: {error_put*100:.2}%')

G-LSM call price: 3.1e+02
G-LSM put price: 3.8e+02
Call error: 0.82%
Put error: 4.5%


# Pricing options on assets with dividends

This changes things, now we can not use the analyitcal solution to the Black-Scholes model to find our desired results. We instead use a tree based approach to simulate the analytical price path and make the correct decision at each time step. Our G-LSM model does not need to be adjusted at all.

In [21]:
import numpy as np

def bermudan_basket_benchmark(p, shrunk_cov_annual):
    # 1. Reduction Parameters
    d = p['dim']
    S0_geo = np.exp(np.mean(np.log(p['S0'])))
    
    # Calculate reduced sigma from the full covariance matrix
    sigma_hat = np.sqrt(np.sum(shrunk_cov_annual)) / d
    
    # Calculate reduced dividend yield (delta_hat)
    sigmas_sqr = np.diag(shrunk_cov_annual)
    # yields_04032026 is your individual dividend yield array
    delta_hat = np.mean(p['dividend_array'] + 0.5 * sigmas_sqr) - 0.5 * sigma_hat**2
    
    # 2. Tree Parameters
    N = p['numTimeStep'] # Match your G-LSM steps (e.g., 50)
    T = p['expiration']
    r = p['rate']
    K = p['strike']
    dt = T / N
    
    u = np.exp(sigma_hat * np.sqrt(dt))
    d_down = 1 / u
    p_up = (np.exp((r - delta_hat) * dt) - d_down) / (u - d_down)
    df = np.exp(-r * dt)
    
    # 3. Initialize Tree at Maturity
    S = S0_geo * (u ** np.arange(N, -N - 1, -2))
    if p['callput'] == 'call':
        V = np.maximum(S - K, 0)
    else:
        V = np.maximum(K - S, 0)
        
    # 4. Backward Induction with Bermudan Exercise
    for i in range(N - 1, -1, -1):
        # Continuation Value
        V = df * (p_up * V[:-1] + (1 - p_up) * V[1:])
        
        # Stock price at this node
        S = S0_geo * (u ** np.arange(i, -i - 1, -2))
        
        # Immediate Exercise Value
        if p['callput'] == 'call':
            EV = np.maximum(S - K, 0)
        else:
            EV = np.maximum(K - S, 0)
            
        # Bermudan Decision
        V = np.maximum(V, EV)
        
    return V[0]


Selecting 8 stocks in the sp500 and calculating their covariance using the LedoitWolf method

In [22]:
start = '2023-09-24'
end = '2026-03-04'
tickers = ['AAPL', 'TSLA', 'LLY', 'BRK-B', 'V', 'BAC', 'WFC', 'IBM']

def create_cov(tickers, start, end):
    # download the data
    prices = yf.download(tickers, start, end, auto_adjust=False)['Adj Close']

    # 1. Prepare your data (dropping NAs is crucial for the solver)
    log_returns = np.log(prices / prices.shift(1)).dropna()

    # 2. Initialize the Ledoit-Wolf estimator
    lw = LedoitWolf()

    # 3. Fit the model to your returns 
    # Note: sklearn expects (observations, features), which matches your log_returns layout
    lw.fit(log_returns)

    # 4. Extract the shrunKcovariance matrix
    # This is the "cleaner" version of your cov_matrix
    shrunk_cov = lw.covariance_

    # 5. Annualize (same as your original code)
    shrunk_cov_annual = shrunk_cov * 252

    prices.to_csv("data/stock_prices_04032026.csv")
    pd.DataFrame(shrunk_cov_annual, columns=tickers, index=tickers).to_csv("data/covariance_matrix_04032026.csv")


shrunk_cov_annual = np.array(pd.read_csv("data/covariance_matrix_04032026.csv", index_col=0))
prices = pd.read_csv("data/stock_prices_04032026.csv", index_col='Date')

In [23]:
# get current prices
S0 = prices.iloc[-1, :]
S0 = np.array(S0)

# get geometric average of the stocKprices
S0_geo_mean = gmean(S0)

# choose strike price
K = 32.5/40 * S0_geo_mean

print(f'Strike: {K}')

Strike: 204.79849399273994


Get the dividend yields for each stock

In [24]:
# yields = []
# for t in tickers:
#     info = yf.Ticker(t).info
#     # Use trailingAnnualDividendYield or dividendYield
#     y = info.get('dividendYield', 0) 
#     yields.append(y if y is not None else 0)

# yields[-1] = 2.68 # for some reason yfinance didn't get the right divident yield for IBM...
# yields

yields_04032026 = [0.39, 0, 0.62, 0, 0.84, 2.24, 2.18, 2.68]

Create parameter dictionary

In [29]:
r = 0.0375             # current BOE rate
div = np.mean(yields_04032026)  # arithmetic mean of dividend yield
T = 0.25               # exercise time
d = len(tickers)       # number of assets

p_call_d = {}
p_call_d['strike'] = K
p_call_d['rate'] = r     
p_call_d['dividend'] = div
p_call_d['dividend_array'] = yields_04032026
p_call_d['expiration'] = T
p_call_d['dim'] = d
p_call_d['S0'] = S0
p_call_d['volatility'] = None
p_call_d['correlation'] = None
p_call_d['covariance'] = shrunk_cov_annual
p_call_d['numTimeStep'] = 50
p_call_d['callput'] = 'call'

p_put_d = p_call_d.copy()
p_put_d['callput'] = 'put'

In [30]:
# Call the main function
glsm_call_d, _ = glsm_geobasketcall.main(p_call_d)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 45.9630
---------------------------------------------
run trial no.2, price = 46.1110
---------------------------------------------
run trial no.3, price = 45.8560
---------------------------------------------
run trial no.4, price = 45.8637
---------------------------------------------
run trial no.5, price = 45.9506
---------------------------------------------
run trial no.6, price = 45.6672
---------------------------------------------
run trial no.7, price = 45.7958
---------------------------------------------
run trial no.8, price = 45.6892
---------------------------------------------
run trial no.9, price = 45.9568
---------------------------------------------
run trial no.10, price = 45.9655
---------------------------------------------

Mean price: 45.8819
Std dev:    0.1296


In [31]:
# Call the main function
glsm_put_d, _ = glsm_geobasketcall.main(p_put_d)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 16.3676
---------------------------------------------
run trial no.2, price = 15.8764
---------------------------------------------
run trial no.3, price = 17.1939
---------------------------------------------
run trial no.4, price = 17.2868
---------------------------------------------
run trial no.5, price = 16.0713
---------------------------------------------
run trial no.6, price = 16.5269
---------------------------------------------
run trial no.7, price = 15.9287
---------------------------------------------
run trial no.8, price = 16.1403
---------------------------------------------
run trial no.9, price = 15.8172
---------------------------------------------
run trial no.10, price = 16.4115
---------------------------------------------

Mean price: 16.3621
Std dev:    0.4930


In [32]:
# accurate estimate with binomial tree model
benchmark_call = bermudan_basket_benchmark(p_call_d, shrunk_cov_annual)
benchmark_put = bermudan_basket_benchmark(p_put_d, shrunk_cov_annual)

print(f'Benchmark call: {benchmark_call}')
print(f'Benchmark put: {benchmark_put}')


Benchmark call: 47.26119092140152
Benchmark put: 15.895866079491455


In [34]:
error_call_d = abs(benchmark_call - glsm_call_d) / abs(benchmark_call)
error_put_d = abs(benchmark_put - glsm_put_d) / abs(benchmark_put)

print(f'G-LSM call price: {glsm_call_d:.2}')
print(f'G-LSM put price: {glsm_put_d:.2}')

print(f'Call error: {error_call_d*100:.2}%')
print(f'Put error: {error_put_d*100:.2}%')

G-LSM call price: 4.6e+01
G-LSM put price: 1.6e+01
Call error: 2.9%
Put error: 2.9%


# Test the accuracy of the model vs moneyness

Preliminary testing suggests that this model may have larger errors when the strike price is far away from the average stock price. We will use the no dividend data to test the accuracy depending on the moneyness of the option, i.e. how far away the strike price is from the average initial stock price.

In [53]:
etf_shrunk_cov_annual = np.array(pd.read_csv("data/etf_covariance_matrix_04032026.csv", index_col=0))
etf_prices = pd.read_csv("data/etf_prices_04032026.csv", index_col='Date')

# get current prices
S0_etf = etf_prices.iloc[-1, :]
S0_etf = np.array(S0_etf)

# get geometric average of the stocKprices
S0_etf_geo_mean = gmean(S0_etf)


r = 0.0375             # current BOE rate
div = 0                # no dividends on ETFs
T = 0.25               # exercise time
d = len(tickers)       # number of assets

p_call = {}
p_call['rate'] = r     
p_call['dividend'] = div
p_call['expiration'] = T
p_call['dim'] = d
p_call['S0'] = S0_etf
p_call['volatility'] = None
p_call['correlation'] = None
p_call['covariance'] = etf_shrunk_cov_annual
p_call['numTimeStep'] = 50
p_call['callput'] = 'call'

p_put = p_call.copy()
p_put['callput'] = 'put'

# create df to store results
cols=["strike_mul", "strike", "BS_call", "BS_put", "glsm_call", "glsm_put"]
results_df = pd.DataFrame(columns=cols)

for m in np.linspace(0.9, 1.15, 8):
    # choose strike price
    K_etf = m * S0_etf_geo_mean

    p_call['strike'] = K_etf
    p_put['strike'] = K_etf

    C, P = put_call_parity(etf_shrunk_cov_annual, S0_etf_geo_mean, K_etf, r, d, T)

    # Call the main function
    glsm_c, std_c = glsm_geobasketcall.main(p_call)
    glsm_p, std_p = glsm_geobasketcall.main(p_put)

    res = [m, K_etf, C, P, glsm_c, glsm_p]
    results_df.loc[len(results_df)] = res


results_df.to_csv("data/moneyness_test.csv")
results_df

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 1751.3922
---------------------------------------------
run trial no.2, price = 1706.7881
---------------------------------------------
run trial no.3, price = 1760.0367
---------------------------------------------
run trial no.4, price = 1701.3804
---------------------------------------------


KeyboardInterrupt: 